In [ ]:
!pip install flaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 83.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [flaml]


# Downgrade com conda

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

✨🍰✨ Everything looks OK!


In [ ]:
!conda create -n compautoml python=3.9 auto-sklearn scikit-learn pandas -y

Channels:
 - conda-forge
Platform: linux-64
Solving environment: \ | / - done

## Package Plan ##

  environment location: /usr/local/envs/compautoml

  added / updated specs:
    - auto-sklearn
    - pandas
    - python=3.9
    - scikit-learn


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    auto-sklearn-0.15.0        |     pyha804496_1         648 KB  conda-forge
    aws-c-auth-0.9.1           |       h194c533_5         120 KB  conda-forge
    aws-c-cal-0.9.8            |       h346e085_0          55 KB  conda-forge
    aws-c-common-0.12.5        |       hb03c661_1         233 KB  conda-forge
    aws-c-compression-0.3.1    |       h7e655bb_8          22 KB  conda-forge
    aws-c-event-stream-0.5.6   |       h1deb5b9_4          58 KB  conda-forge
    aws-c-http-0.10.7          |       had4b759_1         219 KB  conda-forge
    aws-c-io-0.23.2            |       hbff472d_2       

In [ ]:
!python --version
!conda run -n compautoml python --version
!conda info

Python 3.13.13
Python 3.9.23

     active environment : None
       user config file : /root/.condarc
 populated config files : /usr/local/.condarc
          conda version : 26.3.2
    conda-build version : not installed
         python version : 3.13.13.final.0
                 solver : libmamba (default)
       virtual packages : __archspec=1=skylake_avx512
                          __conda=26.3.2=0
                          __cuda=13.0=0
                          __glibc=2.39=0
                          __linux=6.6.159=0
                          __unix=0=0
       base environment : /usr/local  (writable)
      conda av data dir : /usr/local/etc/conda
  conda av metadata url : None
           channel URLs : https://conda.anaconda.org/conda-forge/linux-64
                          https://conda.anaconda.org/conda-forge/noarch
          package cache : /usr/local/pkgs
                          /root/.conda/pkgs
       envs directories : /usr/local/envs
                          /root/

# Importando as Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import load_iris

# Carregando o dataset

In [ ]:
iris = load_iris()

x = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species_id")


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print(f'''
Treino de x: {x_train.shape}
Teste de x: {x_test.shape}
Treino de y: {y_train.value_counts()}
Teste de y: {y_test.value_counts()}''')


Treino de x: (120, 4)
Teste de x: (30, 4)
Treino de y: species_id
0    40
2    40
1    40
Name: count, dtype: int64
Teste de y: species_id
0    10
2    10
1    10
Name: count, dtype: int64


# Flaml

In [ ]:
from flaml import AutoML

In [ ]:
start_flaml = time.time()

automl = AutoML()
automl.fit(X_train=x_train, y_train=y_train, task="classification", metric="accuracy", time_budget=120)

time_flaml = time.time() - start_flaml

y_pred_flaml = automl.predict(x_test)

acc_flaml = accuracy_score(y_test, y_pred_flaml)

print(f'''
\nTempo de execução: {time_flaml}
Melhor modelo FLAML: {automl.best_estimator}
Acurácia: {acc_flaml:.2f}
Relatório de classificação:
{classification_report(y_test, y_pred_flaml)}''')

[flaml.automl.logger: 11-12 20:07:34] {2484} INFO - task = classification
[flaml.automl.logger: 11-12 20:07:34] {2495} INFO - Evaluation method: cv
[flaml.automl.logger: 11-12 20:07:34] {2604} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 11-12 20:07:34] {2721} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 11-12 20:07:34] {3026} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 11-12 20:07:34] {3161} INFO - Estimated sufficient time budget=2416s. Estimated necessary time budget=56s.
[flaml.automl.logger: 11-12 20:07:34] {3212} INFO -  at 0.3s,	estimator lgbm's best error=5.8333e-02,	best estimator lgbm's best error=5.8333e-02
[flaml.automl.logger: 11-12 20:07:34] {3026} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 11-12 20:07:34] {3212} INFO -  at 0.5s,	estimator lgbm's best error=5.8333e-02,	best estimator lgbm's best error=5.8333e-02
[flaml.auto

# Auto-sklearn

In [ ]:
#import autosklearn.classification
#!conda run -n compautoml python -c "import autosklearn.classification"

/usr/local/envs/compautoml/lib/python3.9/site-packages/autosklearn/__init__.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
#!conda run -n compautoml python - << 'EOF'
%%writefile treino_askl.py

import joblib
import time
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import autosklearn.classification

iris = load_iris()

x = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species_id")

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print(f'''
Treino de x: {x_train.shape}
Teste de x: {x_test.shape}
Treino de y: {y_train.value_counts()}
Teste de y: {y_test.value_counts()}''')


start_autosk = time.time()

autosk = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=120,
    per_run_time_limit=10,
)

autosk.fit(x_train, y_train)
time_autosk = time.time() - start_autosk

y_pred_autosk = autosk.predict(x_test)
acc_autosk = accuracy_score(y_test, y_pred_autosk)

print(f'Tempo Auto-sklearn: {time_autosk}')
print(f'Acurácia auto-sklearn: {acc_autosk}')
print(f'Relatório de classificação: \n{classification_report(y_test, y_pred_autosk)}')
print(autosk.leaderboard())

Overwriting treino_askl.py


In [ ]:
!conda run -n compautoml python treino_askl.py


Treino de x: (120, 4)
Teste de x: (30, 4)
Treino de y: species_id
0    40
2    40
1    40
Name: count, dtype: int64
Teste de y: species_id
0    10
2    10
1    10
Name: count, dtype: int64
[WARNING] [2026-11-12 19:51:07,876:smac.runhistory.runhistory2epm.RunHistory2EPM4LogCost] Got cost of smaller/equal to 0. Replace by 0.000010 since we use log cost.
[WARNING] [2026-11-12 19:51:09,346:smac.runhistory.runhistory2epm.RunHistory2EPM4LogCost] Got cost of smaller/equal to 0. Replace by 0.000010 since we use log cost.
[WARNING] [2026-11-12 19:51:14,102:smac.runhistory.runhistory2epm.RunHistory2EPM4LogCost] Got cost of smaller/equal to 0. Replace by 0.000010 since we use log cost.
[WARNING] [2026-11-12 19:51:15,055:smac.runhistory.runhistory2epm.RunHistory2EPM4LogCost] Got cost of smaller/equal to 0. Replace by 0.000010 since we use log cost.
[WARNING] [2026-11-12 19:51:16,336:smac.runhistory.runhistory2epm.RunHistory2EPM4LogCost] Got cost of smaller/equal to 0. Replace by 0.000010 since we

# Relatório

O autosklearn apresentou melhor acurácia geral, com 0.96 de precisão contra 0.90 do flaml. A acurácia por classe também foi mais alta no autosklearn em comparação ao flaml, apresentando duas classes com 100% de precisão enquanto o flaml apresentou somente uma classe com 100%.

O flaml registrou um tempo de execução total de 120.02 segundos contra 114.08 do autosklearn.

O autosklearn apresenta uma forma mais simples de visualização para inspeção do modelo escolhido, através do ranking, podendo visualizar os melhores modelos e as taxas de erros facilmente.
